# Discesa stocastica del gradiente

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

Nelle prime due lezioni abbiamo imparato a costruire reti completamente connesse sovrapponendo strati dense. Quando la rete viene creata, tutti i pesi sono impostati casualmente: la rete non «sa» ancora nulla. In questa lezione vedremo come addestrare una rete neurale: vedremo come le reti neurali *imparano*.

Come per tutti i compiti di machine learning, cominciamo da un insieme di dati di training. Ogni esempio contiene alcune feature, gli ingressi, insieme al target atteso, l’uscita. Addestrare la rete significa regolare i pesi in modo che possa trasformare le feature nel target. Nel dataset *80 Cereals*, per esempio, vogliamo una rete che riceva il contenuto di `'sugar'`, `'fiber'` e `'protein'` di ogni cereale e produca una previsione delle sue `'calories'`. Se riusciamo ad addestrarla, i suoi pesi devono rappresentare in qualche modo la relazione tra quelle feature e quel target così come si presenta nei dati di training.

Oltre ai dati di training, ci servono altre due cose:

- Una «funzione di perdita» che misuri quanto siano buone le previsioni della rete.
- Un «ottimizzatore» che indichi alla rete come modificare i pesi.

# La funzione di perdita

Abbiamo visto come progettare l’architettura di una rete, ma non come comunicarle *quale* problema deve risolvere. È questo il compito della funzione di perdita.

La **funzione di perdita**, o *loss function*, misura la differenza tra il valore reale del target e quello previsto dal modello.

Problemi diversi richiedono funzioni di perdita diverse. Finora abbiamo considerato problemi di **regressione**, nei quali si prevede un valore numerico: le calorie in *80 Cereals*, il giudizio di qualità in *Red Wine Quality*. Altri compiti di regressione potrebbero essere prevedere il prezzo di una casa o l’efficienza nei consumi di un’automobile.

Una funzione di perdita comune per la regressione è l’**errore assoluto medio**, o **MAE**. Per ogni previsione `y_pred`, il MAE misura lo scostamento dal target reale `y_true` mediante la differenza assoluta `abs(y_true - y_pred)`.

La perdita MAE complessiva su un dataset è la media di tutte queste differenze assolute.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/VDcvkZN.png" width="500" alt="Grafico con segmenti di errore tra i punti dei dati e la retta adattata.">
<figcaption><center>L’errore assoluto medio è la lunghezza media dei segmenti tra la curva adattata e i punti dei dati.</center></figcaption>
</figure>

Oltre al MAE, altre funzioni che puoi incontrare nella regressione sono l’errore quadratico medio, o MSE, e la perdita di Huber, entrambe disponibili in Keras.

Durante l’addestramento, il modello usa la funzione di perdita come guida per trovare i valori corretti dei pesi: una perdita minore è migliore. In altre parole, la funzione di perdita indica alla rete il suo obiettivo.

# L’ottimizzatore: discesa stocastica del gradiente

Abbiamo descritto il problema da risolvere, ma ora dobbiamo dire alla rete *come* risolverlo. È questo il compito dell’**ottimizzatore**, un algoritmo che regola i pesi per minimizzare la perdita.

Praticamente tutti gli algoritmi di ottimizzazione usati nel deep learning appartengono alla famiglia della **discesa stocastica del gradiente**, o SGD. Sono algoritmi iterativi che addestrano una rete per passi. Un **passo** di addestramento funziona così:

1. Si campionano alcuni dati di training e li si fa attraversare la rete per ottenere previsioni.
2. Si misura la perdita tra le previsioni e i valori reali.
3. Infine, si regolano i pesi in una direzione che riduce la perdita.

Si ripete questo procedimento finché la perdita non è abbastanza piccola, oppure finché non diminuisce più.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/rFI1tIk.gif" width="1600" alt="Adattamento di una retta un batch alla volta: la perdita diminuisce e i pesi si avvicinano ai valori corretti.">
<figcaption><center>Addestramento di una rete neurale con la discesa stocastica del gradiente.</center></figcaption>
</figure>

Il campione di dati di training usato in ogni iterazione si chiama **minibatch**, o spesso soltanto «batch». Un passaggio completo attraverso i dati di training si chiama **epoca**, o *epoch*. Il numero di epoche di addestramento indica quante volte la rete vedrà ciascun esempio.

L’animazione mostra il modello lineare della prima lezione mentre viene addestrato con SGD. I punti rosso chiaro rappresentano l’intero training set, mentre i punti rosso pieno sono i minibatch. Ogni volta che SGD vede un nuovo minibatch, sposta i pesi, `w` per il coefficiente angolare e `b` per l’intercetta, verso i valori corretti per quel batch. Batch dopo batch, la retta converge infine al miglior adattamento. Puoi vedere che la perdita diminuisce man mano che i pesi si avvicinano ai valori corretti.

## Tasso di apprendimento e dimensione del batch

Nota che la retta compie solo un piccolo spostamento nella direzione indicata da ogni batch, anziché spostarsi completamente. L’ampiezza di questi spostamenti è determinata dal **tasso di apprendimento**, o *learning rate*. Un tasso più piccolo significa che la rete deve vedere più minibatch prima che i pesi convergano ai valori migliori.

Il tasso di apprendimento e la dimensione dei minibatch sono i due parametri che influenzano maggiormente l’andamento dell’addestramento SGD. La loro interazione è spesso delicata e la scelta corretta non è sempre evidente. Esploreremo questi effetti nell’esercizio.

Fortunatamente, per la maggior parte dei lavori non sarà necessaria una ricerca estesa degli iperparametri per ottenere risultati soddisfacenti. **Adam** è un algoritmo SGD con un tasso di apprendimento adattivo, adatto alla maggior parte dei problemi senza regolare i parametri: in un certo senso «si regola da solo». Adam è un ottimo ottimizzatore di uso generale.

## Aggiungere perdita e ottimizzatore

Dopo aver definito il modello, puoi aggiungere una funzione di perdita e un ottimizzatore con il metodo `compile`:

```
model.compile(
    optimizer="adam",
    loss="mae",
)
```

Nota che possiamo specificare perdita e ottimizzatore usando soltanto una stringa. Puoi accedervi anche direttamente tramite l’API di Keras, per esempio per regolare dei parametri; per noi, però, i valori predefiniti andranno bene.

> **Che cosa significa questo nome?**
>
> Il **gradiente** è un vettore che indica in quale direzione devono muoversi i pesi. Più precisamente, indica come modificare i pesi per far cambiare la perdita *più rapidamente*. Chiamiamo il processo **discesa** del gradiente perché usa il gradiente per *scendere* lungo la curva della perdita verso un minimo. **Stocastico** significa «determinato dal caso»: l’addestramento è stocastico perché i minibatch sono *campioni casuali* del dataset. Ecco perché si chiama SGD!


# Esempio: Red Wine Quality

Ora sappiamo tutto ciò che serve per iniziare ad addestrare modelli di deep learning. Vediamolo in azione! Useremo il dataset *Red Wine Quality*.

Questo dataset contiene misure fisico-chimiche relative a circa 1.600 vini rossi portoghesi. Per ogni vino è inclusa anche una valutazione della qualità ottenuta da degustazioni alla cieca. Quanto bene possiamo prevedere la qualità percepita di un vino a partire da queste misure?

Abbiamo raccolto tutta la preparazione dei dati nella prossima cella nascosta. Non è essenziale per ciò che segue, quindi puoi saltarla. Per ora, però, nota che abbiamo riscalato ogni feature nell’intervallo $[0, 1]$. Come approfondiremo nella lezione 5, le reti neurali tendono a funzionare meglio quando gli ingressi hanno una scala comune.


In [ ]:

import pandas as pd
from IPython.display import display

red_wine = pd.read_csv('../input/dl-course-data/red-wine.csv')

# Suddividi i dati in training e validazione.
df_train = red_wine.sample(frac=0.7, random_state=0)
df_valid = red_wine.drop(df_train.index)
display(df_train.head(4))

# Riporta i valori nell'intervallo [0, 1].
max_ = df_train.max(axis=0)
min_ = df_train.min(axis=0)
df_train = (df_train - min_) / (max_ - min_)
df_valid = (df_valid - min_) / (max_ - min_)

# Separa feature e target.
X_train = df_train.drop('quality', axis=1)
X_valid = df_valid.drop('quality', axis=1)
y_train = df_train['quality']
y_valid = df_valid['quality']

Quanti ingressi dovrebbe avere questa rete? Possiamo scoprirlo guardando il numero di colonne della matrice dei dati. Assicurati di non includere il target, `'quality'`: considera soltanto le feature in ingresso.


In [ ]:
print(X_train.shape)

Undici colonne significano undici ingressi.

Abbiamo scelto una rete a tre strati con oltre 1.500 neuroni. Questa rete dovrebbe essere in grado di apprendere relazioni piuttosto complesse nei dati.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=[11]),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])

La scelta dell’architettura del modello dovrebbe far parte di un processo. Parti da una soluzione semplice e lasciati guidare dalla perdita di validazione. Negli esercizi approfondirai lo sviluppo dei modelli.

Dopo aver definito il modello, lo compiliamo specificando l’ottimizzatore e la funzione di perdita.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)

Ora siamo pronti per iniziare l’addestramento! Abbiamo detto a Keras di fornire all’ottimizzatore 256 righe di dati di training alla volta, tramite `batch_size`, e di ripetere dieci volte il passaggio completo attraverso il dataset, tramite `epochs`.


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=256,
    epochs=10,
)

Puoi vedere che Keras ti tiene aggiornato sulla perdita durante l’addestramento.

Spesso, però, un modo migliore per osservare la perdita è tracciarne il grafico. Il metodo `fit` registra infatti la perdita prodotta durante l’addestramento in un oggetto `History`. Convertiremo questi dati in un DataFrame pandas, che rende facile creare il grafico.


In [ ]:
import pandas as pd

# Converti lo storico del training in un DataFrame.
history_df = pd.DataFrame(history.history)
# Usa il metodo plot di pandas.
history_df['loss'].plot();

Nota come la perdita si stabilizza con il passare delle epoche. Quando la curva diventa orizzontale in questo modo, significa che il modello ha imparato tutto ciò che può e non ci sarebbe motivo di continuare per altre epoche.


# Ora tocca a te

Ora [**usa la discesa stocastica del gradiente**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex3.html) per addestrare la tua rete.
